<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/hallucination_filter_extension_Local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Requirements
!pip install sentence-transformers jsonlines rouge_score transformers mistralai bert-score  -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.9/754.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is 

In [ ]:
#@title Setup: Mount Drive and verify paths
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
shared_path = "/content/drive/MyDrive/ColabNotebooks/SigExt/experiments" #@param{type:"string"}
print(os.path.exists(shared_path))


base_path = shared_path

# check all needed files exist
files_to_check = [
    f"{base_path}/cnn_dataset_with_keyphrase/test.jsonl",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl",
    f"{base_path}/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json",
]

all_ok = True
for f in files_to_check:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'}  {f}")
    if not exists:
        all_ok = False

print(f"\nAll files present: {all_ok}")


Mounted at /content/drive
True
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_with_keyphrase/test.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json

All files present: True


In [ ]:
#@title Clone repository
#@markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"

In [ ]:
#@title Imports and Configuration
import os
import json
import nltk #imports the Natural Language Toolkit used to split text into sentences
import torch
import numpy as np
import tqdm #progress bar library
import jsonlines
import nltk
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util #util provides the cosine similarity
from transformers import pipeline as hf_pipeline #provides AI models


%cd {shared_path}
nltk.download('stopwords')
nltk.download('punkt_tab')


#define all paths
base = shared_path
path_scored_dataset   = f"{base}/cnn_dataset_with_keyphrase/test.jsonl" #Path to the test articles that have phrase scores added by the Longformer
path_predictions      = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_predictions.json" #Path to the 500 raw summaries generated by Mistral
path_full_dataset     = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_dataset.jsonl" #Path to the full dataset file that includes raw_output
path_baseline_metrics = f"{base}/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json" #Path to the ROUGE scores that were computed by the original SigExt pipeline
path_output           = f"{base}/hallucination_extension/cnn_verified_predictions/" #Path to the folder where the extension will save its results

os.makedirs(path_output, exist_ok=True)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


/content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt/experiments
Device: cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#@title Load Models
print("Loading sentence embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2") #Downloads and loads the sentence embedding model, used to produce meaningful sentence embeddings
embedder = embedder.to(device)

print("Loading NLI model...")
#Instead of manually tokenizing input, running the model, and decoding output, the pipeline handles everything automatically
nli_model = hf_pipeline(
    "text-classification",
    model="dleemiller/ModernCE-large-nli",
    device=0 if device == "cuda" else -1,
    top_k=None,
    batch_size=32,
    truncation=True,
    max_length=512,
)


Loading sentence embedder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
#@title LLM Backend Selection
# ─────────────────────────────────────────────────────────────────────────────
# Choose which backend to use for sentence regeneration during the pipeline.
#
#   "api"   → Mistral cloud API  (fast but subject to rate limits on free tier)
#   "local" → Mistral-7B loaded in 4-bit directly on the Colab GPU
#              (no API calls, no rate limits, ~1-2 GB VRAM, slower first call)
# ─────────────────────────────────────────────────────────────────────────────

REGEN_BACKEND = "local"  #@param ["api", "local"]

print(f"Regeneration backend selected: {REGEN_BACKEND}")
print("Run Cell 5A (API) or Cell 5B (Local) to initialise the chosen backend.")

In [ ]:
#@title 5A — LLM for Regeneration: Mistral API
# Run this cell ONLY if REGEN_BACKEND == "api"
# The free-tier Mistral API enforces a rate limit of ~1 req/s (≈ 60 req/min).
# During Optuna tuning (50 trials × ~100 validation examples × up to 2 regen
# attempts) this easily triggers HTTP 429 errors, which is why the local
# backend (Cell 5B) is strongly recommended for tuning.

import sys
import time
from mistralai.client import Mistral  # pip install mistralai

MISTRAL_API_KEY = "YdbLhqEepgXav0x8ruTJiWR20GsgWCrD"  #@param {type:"string"}


def predict_one_eg_mistral_api(example, max_retries=5, sleep_seconds=10):
    """
    Calls the Mistral cloud API with exponential-back-off retry logic.

    Args:
        example (dict): Must contain the key 'prompt_input' (str).
        max_retries (int): Number of attempts before giving up.
        sleep_seconds (int): Base sleep time on rate-limit errors.

    Returns:
        str | None: Generated text, or None on unrecoverable failure.

    Notes:
        - HTTP 429 (Too Many Requests) is retried with a growing sleep.
        - Any other exception aborts immediately and returns None.
        - The client is instantiated per-call so this function is thread-safe.
    """
    client = Mistral(api_key=MISTRAL_API_KEY)
    wait = sleep_seconds

    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model="mistral-small-latest",
                messages=[{"role": "user", "content": example["prompt_input"]}],
            )
            return response.choices[0].message.content

        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print(f"  [API] Rate limited — attempt {attempt+1}/{max_retries}. "
                      f"Sleeping {wait}s...")
                time.sleep(wait)
                wait = min(wait * 2, 60)   # exponential back-off, cap at 60s
            else:
                print(f"  [API] Non-retriable error: {e}")
                return None

    print(f"  [API] Gave up after {max_retries} attempts.")
    return None


# ── Wire the active predict function ────────────────────────────────────────
if REGEN_BACKEND == "api":
    predict_fn_active = predict_one_eg_mistral_api
    print("✓  Mistral API backend ready.")
    print("   WARNING: free-tier rate limits will slow down Optuna tuning.")
    print("   Consider switching to REGEN_BACKEND = 'local' for tuning runs.")
else:
    print("ℹ  REGEN_BACKEND is not 'api' — skipping API initialisation.")
    print("   Run Cell 5B to load the local model.")

In [ ]:
#@title 5B — LLM for Regeneration: Local Mistral-7B (4-bit, no API calls)
# Run this cell ONLY if REGEN_BACKEND == "local"

import subprocess, sys, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── 0. Ensure bitsandbytes is up to date ────────────────────────────────────
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1"],
    check=True,
)

LOCAL_MODEL_NAME     = "mistralai/Mistral-7B-Instruct-v0.3"  #@param {type:"string"}
LOCAL_MAX_NEW_TOKENS = 128   #@param {type:"integer"}
LOCAL_TEMPERATURE    = 0.0   #@param {type:"number"}

# ── 1. 4-bit quantisation config ────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# ── 2. Tokenizer ─────────────────────────────────────────────────────────────
print(f"Loading tokenizer for {LOCAL_MODEL_NAME} ...")
local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
if local_tokenizer.pad_token is None:
    local_tokenizer.pad_token = local_tokenizer.eos_token

# ── 3. Model ─────────────────────────────────────────────────────────────────
print(f"Loading {LOCAL_MODEL_NAME} in 4-bit NF4 quantisation ...")
local_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
local_model.eval()
print(f"Model loaded. VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


# ── 4. Inference function ─────────────────────────────────────────────────────
def predict_one_eg_local(example,
                          max_new_tokens=LOCAL_MAX_NEW_TOKENS,
                          temperature=LOCAL_TEMPERATURE):
    """
    Runs local Mistral-7B inference for one regeneration prompt.

    Uses tokenize=False in apply_chat_template to obtain a plain string,
    then tokenizes separately — this avoids BatchEncoding inconsistencies
    across transformers versions (pattern from DNLPsigext working notebook).

    Args:
        example (dict): Must contain key 'prompt_input' (str).
        max_new_tokens (int): Maximum tokens to generate.
        temperature (float): 0 = greedy deterministic, >0 = sampling.

    Returns:
        str | None: Generated rewrite text, or None on failure / empty output.
    """
    try:
        prompt_text = example["prompt_input"]
        messages    = [{"role": "user", "content": prompt_text}]

        # get a plain string — avoids BatchEncoding .shape issues entirely
        prompt_str = local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # tokenize separately to get a clean {input_ids, attention_mask} dict
        inputs = local_tokenizer(
            prompt_str,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(local_model.device)

        prompt_len = inputs["input_ids"].shape[1]

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=local_tokenizer.eos_token_id,
            pad_token_id=local_tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=True,
        )

        if temperature > 0:
            gen_kwargs["do_sample"]   = True
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"]       = 0.9
            del gen_kwargs["temperature"]  # remove None placeholder
            gen_kwargs["temperature"] = temperature

        with torch.no_grad():
            output_ids = local_model.generate(**inputs, **gen_kwargs)

        new_tokens = output_ids[0][prompt_len:]

        if len(new_tokens) == 0:
            print("  [Local] Warning: model generated 0 new tokens.")
            return None

        result = local_tokenizer.decode(
            new_tokens, skip_special_tokens=True
        ).strip()
        result = " ".join(result.replace("\n", " ").split())

        return result if result else None

    except Exception as e:
        import traceback
        print(f"  [Local] Inference error: {type(e).__name__}: {e}")
        traceback.print_exc()
        return None


# ── 5. Sanity check ───────────────────────────────────────────────────────────
print("\nRunning sanity check ...")
_test = predict_one_eg_local({
    "prompt_input": (
        "Rewrite the following sentence using only the evidence provided.\n\n"
        "Evidence: Scientists at CERN confirmed the Higgs boson particle in 2012.\n\n"
        "Original sentence: Researchers discovered a new particle last year.\n\n"
        "Rewrite the sentence faithfully. "
        "If impossible respond with exactly UNSUPPORTED."
    )
})
if _test:
    print(f"✓  Sanity check passed.\n   Output: {_test[:120]}")
else:
    print("✗  Sanity check returned None — check warnings above.")


# ── 6. Wire the active predict function ──────────────────────────────────────
if REGEN_BACKEND == "local":
    predict_fn_active = predict_one_eg_local
    print("\n✓  Local Mistral-7B backend ready.")
    print(f"   max_new_tokens = {LOCAL_MAX_NEW_TOKENS}")
    print(f"   temperature    = {LOCAL_TEMPERATURE}")
    print("   No API calls will be made during tuning or inference.")
else:
    print("ℹ  REGEN_BACKEND is not 'local' — skipping model download.")
    print("   Run Cell 5A to configure the API backend instead.")


In [ ]:
#@title Hyperparameter Tuning — Cached NLI + Embeddings (fast)
!pip install optuna -q

import optuna
import numpy as np
import json
import os
from rouge_score import rouge_scorer
from sentence_transformers import util

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS     = 20
TOP_N_CACHE  = 10   # cache top-10, then slice to top_n during trial

# ── Step 1: Pre-compute and cache all scores ─────────────────────────────────
print("Pre-computing embeddings and NLI scores for validation set ...")
print("This runs once — trials will use cached values.")

val_cache = []   # one entry per validation example

for raw_sum, article in zip(val_summaries, val_dataset):
    example_cache = {
        "raw_summary":   raw_sum,
        "source":        article["trunc_input"],
        "sentences":     [],
    }

    if not raw_sum or not raw_sum.strip():
        val_cache.append(example_cache)
        continue

    summary_sentences = split_sentences(raw_sum)
    source_sentences  = split_sentences(article["trunc_input"])

    if not summary_sentences or not source_sentences:
        val_cache.append(example_cache)
        continue

    # embed all source sentences once per example
    source_embs = embedder.encode(source_sentences, convert_to_tensor=True)

    for sent in summary_sentences:
        sent_emb = embedder.encode(sent, convert_to_tensor=True)

        # cosine similarity scores against all source sentences
        cos_scores = util.cos_sim(sent_emb, source_embs)[0].tolist()

        # get top-10 indices by cosine similarity
        top_idx = sorted(range(len(cos_scores)),
                         key=lambda i: cos_scores[i], reverse=True)[:TOP_N_CACHE]

        top_evidence  = [source_sentences[i] for i in top_idx]
        top_cos       = [cos_scores[i]        for i in top_idx]

        # run NLI on top-10 evidence sentences
        nli_scores = []
        for ev in top_evidence:
            result       = nli_model({"text": ev, "text_pair": sent})
            label_scores = {item["label"]: item["score"] for item in result}
            nli_scores.append({
                "entailment":    label_scores.get("entailment",    0.0),
                "contradiction": label_scores.get("contradiction", 0.0),
                "neutral":       label_scores.get("neutral",       0.0),
            })

        example_cache["sentences"].append({
            "sentence":    sent,
            "evidence":    top_evidence,
            "cos_scores":  top_cos,
            "nli_scores":  nli_scores,
        })

    val_cache.append(example_cache)

print(f"Cache built: {len(val_cache)} examples, "
      f"{sum(len(e['sentences']) for e in val_cache)} total sentences.")


# ── Step 2: Fast evaluation using only cached scores ─────────────────────────
def evaluate_config_cached(top_n, threshold, similarity, max_regen, contra_gate):
    """
    Applies threshold logic on pre-computed NLI + cosine scores.
    Zero model calls — runs in milliseconds per trial.
    """
    final_sums = []
    total_flagged = 0
    total_sents   = 0

    for entry in val_cache:
        raw_sum = entry["raw_summary"]

        if not raw_sum or not raw_sum.strip() or not entry["sentences"]:
            final_sums.append("")
            continue

        verified = []

        for s in entry["sentences"]:
            total_sents += 1

            # slice to top_n
            cos_scores = s["cos_scores"][:top_n]
            nli_scores = s["nli_scores"][:top_n]

            best_entail = max(n["entailment"]    for n in nli_scores)
            best_contra = max(n["contradiction"] for n in nli_scores)
            best_sim    = max(cos_scores)

            # same logic as check_entailment
            if best_entail >= threshold or best_sim >= similarity:
                verified.append(s["sentence"])
            elif best_contra >= 0.7 and best_sim < contra_gate:
                # flagged — drop (no LLM during tuning)
                total_flagged += 1
            else:
                verified.append(s["sentence"])

        final_sums.append(" ".join(verified))

    # ROUGE on validation set
    scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(final_sums, val_refs):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)

    halluc_rate = total_flagged / total_sents if total_sents > 0 else 0

    return {
        "rouge1f":     round(np.mean(r1_f) * 100, 4),
        "rougeLf":     round(np.mean(rl_f) * 100, 4),
        "rouge1r":     round(np.mean(r1_r) * 100, 4),
        "halluc_rate": round(halluc_rate, 4),
        "total_flagged": total_flagged,
        "regen_ok":    0,
        "dropped":     total_flagged,
        "empty":       sum(1 for s in final_sums if not s or not s.strip()),
    }


# ── Step 3: Optuna study ──────────────────────────────────────────────────────
tuning_results = []

def objective(trial):
    top_n       = trial.suggest_categorical("top_n",       [3, 5])
    threshold   = trial.suggest_categorical("threshold",   [0.3, 0.4, 0.5])
    similarity  = trial.suggest_categorical("similarity",  [0.65, 0.75])
    max_regen   = trial.suggest_categorical("max_regen",   [1])
    contra_gate = trial.suggest_categorical("contra_gate", [0.4, 0.6, 0.8])

    metrics = evaluate_config_cached(top_n, threshold, similarity,
                                     max_regen, contra_gate)
    tuning_results.append({
        "top_n": top_n, "threshold": threshold, "similarity": similarity,
        "max_regen": max_regen, "contra_gate": contra_gate, **metrics
    })
    return metrics["rouge1f"]


print(f"\nRunning {N_TRIALS} Optuna trials (cached — no model calls) ...")
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


# ── Step 4: Print results and apply best config ───────────────────────────────
tuning_results.sort(key=lambda x: x["rouge1f"], reverse=True)

print(f"\n{'top_n':<8} {'thresh':<8} {'sim':<8} {'regen':<8} {'gate':<8} "
      f"{'R1-F1':<10} {'RL-F1':<10} {'halluc%':<10} {'dropped':<8}")
print("-" * 80)
for r in tuning_results[:10]:
    print(f"{r['top_n']:<8} {r['threshold']:<8} {r['similarity']:<8} "
          f"{r['max_regen']:<8} {r['contra_gate']:<8} {r['rouge1f']:<10} "
          f"{r['rougeLf']:<10} {r['halluc_rate']*100:<10.1f} {r['dropped']:<8}")

best_params = study.best_params
TOP_N_EVIDENCE       = best_params["top_n"]
ENTAIL_THRESHOLD     = best_params["threshold"]
SIMILARITY_THRESHOLD = best_params["similarity"]
MAX_REGEN_ATTEMPTS   = best_params["max_regen"]
CONTRA_SIM_GATE      = best_params["contra_gate"]

print(f"\nBest configuration applied:")
print(f"  TOP_N_EVIDENCE       = {TOP_N_EVIDENCE}")
print(f"  ENTAIL_THRESHOLD     = {ENTAIL_THRESHOLD}")
print(f"  SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}")
print(f"  MAX_REGEN_ATTEMPTS   = {MAX_REGEN_ATTEMPTS}")
print(f"  CONTRA_SIM_GATE      = {CONTRA_SIM_GATE}")
print(f"  ROUGE-1 F1 (val)     = {tuning_results[0]['rouge1f']}")

os.makedirs(path_output, exist_ok=True)
with open(f"{path_output}tuning_results_optuna.json", "w") as f:
    json.dump(tuning_results, f, indent=2)
print(f"\nAll {len(tuning_results)} results saved to {path_output}tuning_results_optuna.json")

In [ ]:
#@title LLM for Regeneration

import sys
import os
from types import ModuleType
from mistralai.client import Mistral
import tqdm

MISTRAL_API_KEY = "YdbLhqEepgXav0x8ruTJiWR20GsgWCrD"

def predict_one_eg_mistral(example, max_retries=5, sleep_seconds=10):
    """
    Calls Mistral API with retry logic to handle rate limits.
    Returns the generated text or None on failure.
    """
    import time
    client = Mistral(api_key=MISTRAL_API_KEY)

    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model="mistral-small-latest",
                messages=[{"role": "user", "content": example["prompt_input"]}]
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print(f"  Rate limited. Attempt {attempt+1}/{max_retries}. Sleeping {sleep_seconds}s...")
                time.sleep(sleep_seconds)
            else:
                print(f"  API error: {e}")
                return None
    return None

print("Mistral prediction function defined.")

Mistral prediction function defined.


In [ ]:
#@title Load Data

#raw summaries generated by the LLMs
with open(path_predictions) as f:
  raw_summaries = json.load(f)

#source articles with phrase scores
with jsonlines.open(path_scored_dataset) as f:
  scored_dataset = list(f)

#full dataset with raw_output for the ROUGE evaluation
with jsonlines.open(path_full_dataset) as f:
  full_dataset = list(f)

#baseline metrics for comparison
with open(path_baseline_metrics) as f:
  baseline_metrics = json.load(f)

print(f"Summaries loaded:       {len(raw_summaries)}")
print(f"Scored articles loaded: {len(scored_dataset)}")
print(f"Full dataset loaded:    {len(full_dataset)}")
print(f"\nBaseline ROUGE-1 F1: {baseline_metrics.get('rouge1f', 'N/A')}")
print(f"\nSample raw summary:\n{raw_summaries[0]}")

Summaries loaded:       500
Scored articles loaded: 500
Full dataset loaded:    500

Baseline ROUGE-1 F1: 38.8941

Sample raw summary:
A former Japanese school principal, Yuhei Takashima, is facing criminal charges in Tokyo after police arrested him for allegedly paying for sex with over 12,000 women—some as young as 14—in the Philippines, where he also photographed obscene acts and produced pornography. Police seized 147,600 photos documenting his activities, with investigations revealing he traveled to Manila repeatedly since 1988 to "buy sex," citing its affordability.


In [ ]:
#@title Verification Functions

import re
import nltk
import torch
from sentence_transformers import util

# ── Stage 4 — sentence segmentation ──────────────────────────
def split_sentences(text):
    if not text or not text.strip():
        return []
    return [
        s.strip()
        for s in nltk.sent_tokenize(text)
        if s.strip() and len(s.split()) >= 3
    ]


# ── Stage 5 — evidence retrieval ─────────────────────────────
def retrieve_evidence(summary_sentence, source_sentences, top_n=TOP_N_EVIDENCE):
    """
    Retrieves the most semantically similar source sentences as evidence.
    Uses cosine similarity between sentence embeddings.
    Returns top_n most similar source sentences.
    """
    query_emb  = embedder.encode(summary_sentence, convert_to_tensor=True)
    source_emb = embedder.encode(source_sentences, convert_to_tensor=True)
    scores     = util.cos_sim(query_emb, source_emb)
    top_idx    = torch.topk(
        scores, k=min(top_n, len(source_sentences))
    ).indices.flatten()
    return [source_sentences[idx.item()] for idx in top_idx]


# ── Stage 6 — hybrid entailment check ────────────────────────
def check_entailment(summary_sentence, evidence):
    """
    Hybrid faithfulness check combining pairwise NLI and cosine similarity.

    Pass conditions (ANY of these):
        1. NLI entailment >= ENTAIL_THRESHOLD      for any evidence sentence
        2. Cosine similarity >= SIMILARITY_THRESHOLD for any evidence sentence

    Contradiction flag condition (ALL of these must be true):
        3. NLI contradiction >= 0.7
        4. Best cosine similarity < CONTRA_SIM_GATE

    The CONTRA_SIM_GATE prevents false positives where the NLI model
    incorrectly flags correct paraphrases as contradictions.
    A real hallucination has both high contradiction AND low similarity.
    A correct paraphrase confused the NLI model has high contradiction
    but also high similarity because the content is genuinely related.

    Args:
        summary_sentence (str): The summary sentence to verify (hypothesis).
        evidence (list[str]): Source sentences to check against (premises).

    Returns:
        tuple[str, float]: Label and best score found.
    """
    best_entail  = 0.0
    best_contra  = 0.0
    best_neutral = 0.0
    best_sim     = 0.0

    summary_emb = embedder.encode(summary_sentence, convert_to_tensor=True)

    for ev_sentence in evidence:

        # cosine similarity — catches paraphrases that NLI misses
        ev_emb = embedder.encode(ev_sentence, convert_to_tensor=True)
        sim    = float(util.cos_sim(summary_emb, ev_emb)[0][0])
        if sim > best_sim:
            best_sim = sim

        # pairwise NLI — one evidence sentence at a time
        result       = nli_model({"text": ev_sentence, "text_pair": summary_sentence})
        label_scores = {item["label"]: item["score"] for item in result}

        entail  = label_scores.get("entailment",    0.0)
        contra  = label_scores.get("contradiction", 0.0)
        neutral = label_scores.get("neutral",       0.0)

        if entail  > best_entail:  best_entail  = entail
        if contra  > best_contra:  best_contra  = contra
        if neutral > best_neutral: best_neutral = neutral

    # pass: NLI says entailed OR sentences are semantically similar
    if best_entail >= ENTAIL_THRESHOLD or best_sim >= SIMILARITY_THRESHOLD:
        return "entailed", max(best_entail, best_sim)

    # contradiction: only flag if BOTH contradiction is high AND similarity is low
    # high similarity means the sentence is genuinely related to the source
    # and the NLI confusion is likely a paraphrase detection failure
    elif best_contra >= 0.7 and best_sim < CONTRA_SIM_GATE:
        return "contradiction", best_contra

    # everything else passes with benefit of the doubt
    # neutral sentences and high-similarity contradictions are treated as entailed
    else:
        return "entailed", max(best_entail, best_sim)

# ── Stage 7 — sentence regeneration ──────────────────────────
def regenerate_sentence(
    original_sentence,
    evidence,
    predict_fn=predict_one_eg_mistral
):
    """
    Asks the LLM to rewrite a flagged sentence using only the evidence.
    Returns None if the LLM cannot produce a faithful rewrite.
    Keyphrases are NOT passed here to avoid reproducing the same error.
    """
    evidence_text = " ".join(evidence)
    prompt = (
        f"The following sentence may contain information not supported "
        f"by the source document.\n\n"
        f"Evidence from the source:\n{evidence_text}\n\n"
        f"Original sentence:\n{original_sentence}\n\n"
        f"Rewrite the sentence using only information from the evidence. "
        f"Do not add anything not in the evidence. "
        f'If impossible, respond with exactly "UNSUPPORTED".'
    )
    result = predict_fn({"prompt_input": prompt})
    if not result or result.strip().upper() == "UNSUPPORTED":
        return None
    return result.strip()


# ── Stages 4-8 — pipeline orchestrator ───────────────────────
def verify_summary(
    raw_summary,
    source_document,
    predict_fn=predict_one_eg_mistral
):
    """
    Orchestrates the full verification pipeline for one summary.

    For each summary sentence:
        1. Retrieve top-N most similar source sentences as evidence
        2. Check pairwise NLI entailment against each evidence sentence
        3. If passes → keep sentence unchanged
        4. If fails  → attempt regeneration using evidence as constraint
        5. If regeneration passes NLI → use regenerated sentence
        6. If regeneration fails     → drop sentence

    Returns the verified summary and a log of all hallucinations found.
    """
    summary_sentences = split_sentences(raw_summary)
    source_sentences  = split_sentences(source_document)

    if not summary_sentences or not source_sentences:
        return raw_summary, []

    verified   = []
    halluc_log = []

    for i, sentence in enumerate(summary_sentences):

        # Stage 5 — retrieve evidence
        evidence = retrieve_evidence(sentence, source_sentences)

        # Stage 6 — pairwise entailment check
        label, score = check_entailment(sentence, evidence)

        # sentence passes — keep it unchanged
        if label == "entailed":
            verified.append(sentence)
            continue

        # sentence flagged — create audit log entry
        log_entry = {
            "sentence_idx":   i,
            "original":       sentence,
            "label":          label,
            "score":          round(score, 4),
            "evidence":       evidence,
            "regenerated":    None,
            "regen_accepted": False,
        }

        # Stage 7 — attempt regeneration
        regenerated = None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            candidate = regenerate_sentence(sentence, evidence, predict_fn)

            if candidate is None:
                break

            # verify the regenerated sentence with pairwise NLI
            new_label, new_score = check_entailment(candidate, evidence)
            if new_label == "entailed":
                regenerated = candidate
                log_entry["regen_accepted"] = True
                break

        # Stage 8 — finalize log and add to verified if successful
        log_entry["regenerated"] = regenerated
        halluc_log.append(log_entry)

        if regenerated:
            verified.append(regenerated)

    # reassemble final summary
    final_summary = " ".join(verified)
    return final_summary, halluc_log


print("Verification functions defined.")

Verification functions defined.


In [ ]:
#@title Hyperparameter Tuning (Optuna Bayesian Optimisation)
# ─────────────────────────────────────────────────────────────────────────────
# This cell runs Optuna Bayesian optimisation over the five pipeline
# hyperparameters.  The active regeneration backend (api or local, selected
# in the backend selector cell above) is used for every trial.
#
# IMPORTANT: if REGEN_BACKEND == "api" the free Mistral tier will be hit hard
# by 50 trials × up to 100 val examples × up to 2 regen attempts per sentence.
# Switch to REGEN_BACKEND = "local" to avoid rate limits.
# ─────────────────────────────────────────────────────────────────────────────

import optuna
from sklearn.model_selection import ShuffleSplit
from rouge_score import rouge_scorer
import numpy as np
import json
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_SIZE = 500
N_TRIALS  = 50   # increase to 100+ if you have time / local GPU

# 80/20 split: 400 tune / 100 validation
ss = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tune_idx, val_idx = next(ss.split(range(TUNE_SIZE)))

tune_summaries = [raw_summaries[i]  for i in tune_idx]
tune_dataset   = [scored_dataset[i] for i in tune_idx]
val_summaries  = [raw_summaries[i]  for i in val_idx]
val_dataset    = [scored_dataset[i] for i in val_idx]
val_refs       = [full_dataset[i]["raw_output"] for i in val_idx]

print(f"Backend in use for tuning: {REGEN_BACKEND}")
print(f"Validation set size:       {len(val_summaries)}")
print(f"Number of Optuna trials:   {N_TRIALS}\n")


def evaluate_config(top_n, threshold, similarity, max_regen, contra_gate):
    """
    Evaluates one hyperparameter configuration on the validation set.
    Updates all global constants so verification functions pick them up.

    The regeneration function used is always predict_fn_active, which is
    set by whichever backend cell was run last (API or local).
    """
    global TOP_N_EVIDENCE, ENTAIL_THRESHOLD, SIMILARITY_THRESHOLD
    global MAX_REGEN_ATTEMPTS, CONTRA_SIM_GATE

    TOP_N_EVIDENCE       = top_n
    ENTAIL_THRESHOLD     = threshold
    SIMILARITY_THRESHOLD = similarity
    MAX_REGEN_ATTEMPTS   = max_regen
    CONTRA_SIM_GATE      = contra_gate

    final_sums, logs = [], []

    for raw_sum, article in zip(val_summaries, val_dataset):
        if not raw_sum or not raw_sum.strip():
            final_sums.append("")
            logs.append([])
            continue
        # verify_summary will call predict_fn_active internally
        final_sum, halluc_log = verify_summary(
            raw_sum, article["trunc_input"], predict_fn=predict_fn_active
        )
        final_sums.append(final_sum)
        logs.append(halluc_log)

    # ROUGE on validation set
    scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(final_sums, val_refs):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)

    rouge1f = round(np.mean(r1_f) * 100, 4)
    rougeLf = round(np.mean(rl_f) * 100, 4)
    rouge1r = round(np.mean(r1_r) * 100, 4)

    # Hallucination statistics
    total_sents   = sum(len(split_sentences(s)) for s in val_summaries if s)
    total_flagged = sum(len(l) for l in logs)
    regen_ok      = sum(sum(1 for h in log if h["regen_accepted"]) for log in logs)
    halluc_rate   = total_flagged / total_sents if total_sents > 0 else 0
    regen_rate    = regen_ok / total_flagged if total_flagged > 0 else 0
    empty         = sum(1 for s in final_sums if not s or not s.strip())
    dropped       = total_flagged - regen_ok

    return {
        "rouge1f":       rouge1f,
        "rougeLf":       rougeLf,
        "rouge1r":       rouge1r,
        "halluc_rate":   round(halluc_rate, 4),
        "regen_rate":    round(regen_rate, 4),
        "total_flagged": total_flagged,
        "regen_ok":      regen_ok,
        "dropped":       dropped,
        "empty":         empty,
    }


# ── Optuna objective ─────────────────────────────────────────────────────────
tuning_results = []

def objective(trial):
    top_n       = trial.suggest_categorical("top_n",       [3, 5, 7, 10])
    threshold   = trial.suggest_categorical("threshold",   [0.2, 0.3, 0.4, 0.5, 0.6])
    similarity  = trial.suggest_categorical("similarity",  [0.55, 0.65, 0.75, 0.85])
    max_regen   = trial.suggest_categorical("max_regen",   [1, 2])
    contra_gate = trial.suggest_categorical("contra_gate", [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

    metrics = evaluate_config(top_n, threshold, similarity, max_regen, contra_gate)

    tuning_results.append({
        "top_n":       top_n,
        "threshold":   threshold,
        "similarity":  similarity,
        "max_regen":   max_regen,
        "contra_gate": contra_gate,
        "backend":     REGEN_BACKEND,   # record which backend was used
        **metrics
    })

    return metrics["rouge1f"]


print(f"Parameter search space:")
print(f"  top_n:       [3, 5, 7, 10]")
print(f"  threshold:   [0.2, 0.3, 0.4, 0.5, 0.6]")
print(f"  similarity:  [0.55, 0.65, 0.75, 0.85]")
print(f"  max_regen:   [1, 2]")
print(f"  contra_gate: [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]")
print(f"\nStarting Optuna ({N_TRIALS} trials) ...\n")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


# ── Print top-15 results ─────────────────────────────────────────────────────
tuning_results.sort(key=lambda x: x["rouge1f"], reverse=True)

print(f"\n{'top_n':<8} {'thresh':<8} {'sim':<8} {'regen':<8} {'gate':<8} "
      f"{'R1-F1':<10} {'RL-F1':<10} {'halluc%':<10} {'dropped':<8} {'empty':<6}")
print("-" * 88)
for r in tuning_results[:15]:
    print(f"{r['top_n']:<8} {r['threshold']:<8} {r['similarity']:<8} "
          f"{r['max_regen']:<8} {r['contra_gate']:<8} {r['rouge1f']:<10} "
          f"{r['rougeLf']:<10} {r['halluc_rate']*100:<10.1f} "
          f"{r['dropped']:<8} {r['empty']:<6}")


# ── Apply best config ────────────────────────────────────────────────────────
best_params  = study.best_params
best_metrics = tuning_results[0]

TOP_N_EVIDENCE       = best_params["top_n"]
ENTAIL_THRESHOLD     = best_params["threshold"]
SIMILARITY_THRESHOLD = best_params["similarity"]
MAX_REGEN_ATTEMPTS   = best_params["max_regen"]
CONTRA_SIM_GATE      = best_params["contra_gate"]

print(f"\nBest configuration applied:")
print(f"  TOP_N_EVIDENCE       = {TOP_N_EVIDENCE}")
print(f"  ENTAIL_THRESHOLD     = {ENTAIL_THRESHOLD}")
print(f"  SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}")
print(f"  MAX_REGEN_ATTEMPTS   = {MAX_REGEN_ATTEMPTS}")
print(f"  CONTRA_SIM_GATE      = {CONTRA_SIM_GATE}")
print(f"  ROUGE-1 F1 (val)     = {best_metrics['rouge1f']}")
print(f"  Hallucination rate   = {best_metrics['halluc_rate']*100:.1f}%")
print(f"  Dropped              = {best_metrics['dropped']}")
print(f"  Empty summaries      = {best_metrics['empty']}")
print(f"  Backend used         = {best_metrics['backend']}")

# ── Save tuning results ──────────────────────────────────────────────────────
os.makedirs(path_output, exist_ok=True)
with open(f"{path_output}tuning_results_optuna.json", "w") as f:
    json.dump(tuning_results, f, indent=2)
print(f"\nAll {len(tuning_results)} results saved to {path_output}tuning_results_optuna.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.1 MB/s eta 0:00:00
Running Optuna with 50 trials on 100 validation examples...
Parameters being tuned:
  top_n:       [3, 5, 7, 10]
  threshold:   [0.2, 0.3, 0.4, 0.5, 0.6]
  similarity:  [0.55, 0.65, 0.75, 0.85]
  max_regen:   [1, 2]
  contra_gate: [0.3, 0.4, 0.5, 0.6, 0.7]

Estimated time on A100: ~25 minutes



  0%|          | 0/50 [00:00<?, ?it/s]

W0327 09:56:40.662000 5426 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[W 2026-03-27 09:56:52,369] Trial 0 failed with parameters: {'top_n': 5, 'threshold': 0.5, 'similarity': 0.75, 'max_regen': 1, 'contra_gate': 0.5} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_5426/2416625555.py", line 102, in objective
    metrics = evaluate_config(top_n, threshold, similarity, max_regen, contra_gate)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5426/2416625555.py", line 50, in evaluate_config
    final_sum, halluc_log = verify_summary(raw_sum, article["trunc_input"])
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5426/1266420564.py", line 164, in verify_summary
    label, score = check_entailment(sentence, evidence)
          

KeyboardInterrupt: 

In [ ]:
# ── Load best Optuna parameters from saved file ───────────────
optuna_params_path = f"{path_output}tuning_results_optuna.json"

if os.path.exists(optuna_params_path):
    with open(optuna_params_path) as f:
        optuna_results = json.load(f)

    # file is a list sorted by ROUGE-1 F1 descending
    # index 0 is always the best configuration
    best = optuna_results[0]

    TOP_N_EVIDENCE       = best["top_n"]
    ENTAIL_THRESHOLD     = best["threshold"]
    SIMILARITY_THRESHOLD = best["similarity"]
    MAX_REGEN_ATTEMPTS   = best["max_regen"]
    CONTRA_SIM_GATE      = best["contra_gate"]

    print(f"Best Optuna parameters loaded from {optuna_params_path}")
    print(f"\n  TOP_N_EVIDENCE       = {TOP_N_EVIDENCE}")
    print(f"  ENTAIL_THRESHOLD     = {ENTAIL_THRESHOLD}")
    print(f"  SIMILARITY_THRESHOLD = {SIMILARITY_THRESHOLD}")
    print(f"  MAX_REGEN_ATTEMPTS   = {MAX_REGEN_ATTEMPTS}")
    print(f"  CONTRA_SIM_GATE      = {CONTRA_SIM_GATE}")
    print(f"\n  ROUGE-1 F1 (val):    {best['rouge1f']}")
    print(f"  Hallucination rate:  {best['halluc_rate']*100:.1f}%")
    print(f"  Dropped sentences:   {best['dropped']}")

else:
    print(f"File not found: {optuna_params_path}")
    print("\nAvailable files in output directory:")
    for fname in sorted(os.listdir(path_output)):
        fsize = os.path.getsize(f"{path_output}{fname}")
        print(f"  {fname}  ({fsize:,} bytes)")
    print("\nRun Section 8 first to generate the tuning results file.")

Optuna parameters loaded:
  TOP_N_EVIDENCE       = 3
  ENTAIL_THRESHOLD     = 0.6
  SIMILARITY_THRESHOLD = 0.55
  MAX_REGEN_ATTEMPTS   = 1


In [ ]:
#@title Run Verification Pipeline

final_summaries = []
all_halluc_logs  = []


"""
At each iteration:
i = 0
raw_summary = "Scientists confirmed the Higgs boson. The cost was 13 billion."
article     = {
    "trunc_input": "Scientists at CERN confirmed the Higgs boson particle.
                    The research involved 6,000 scientists...",
    "input_kw_model": [...],
    ...
}
"""
for i, (raw_summary, article) in enumerate(
    tqdm.tqdm(
        zip(raw_summaries, scored_dataset),
        total=len(raw_summaries),
        desc="Verifying"
    )
):
    # skip empty summaries
    if not raw_summary or not raw_summary.strip():
        final_summaries.append("")
        all_halluc_logs.append({
            "example_idx":    i,
            "raw_summary":    raw_summary,
            "final_summary":  "",
            "hallucinations": [],
        })
        continue

    source_document = article["trunc_input"]
    final_summary, halluc_log = verify_summary(raw_summary, source_document)

    final_summaries.append(final_summary)
    all_halluc_logs.append({
        "example_idx":    i,
        "raw_summary":    raw_summary,
        "final_summary":  final_summary,
        "hallucinations": halluc_log,
    })

print(f"\nVerification complete.")
print(f"Total examples processed: {len(final_summaries)}")
print(f"Total hallucinations found: {sum(len(l['hallucinations']) for l in all_halluc_logs)}")

Verifying: 100%|██████████| 500/500 [04:58<00:00,  1.67it/s]


Verification complete.
Total examples processed: 500
Total hallucinations found: 18


In [ ]:
#@title  Evaluate and Save Results
from rouge_score import rouge_scorer
import numpy as np
import json
import os

def compute_rouge(predictions, references):
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)
    return {
        "rouge1f": round(np.mean(r1_f) * 100, 4),
        "rougeLf": round(np.mean(rl_f) * 100, 4),
        "rouge1r": round(np.mean(r1_r) * 100, 4),
    }

# human reference summaries
references = [item["raw_output"] for item in full_dataset]

# compute ROUGE for raw and verified summaries
raw_metrics   = compute_rouge(raw_summaries,   references)
final_metrics = compute_rouge(final_summaries, references)

# hallucination statistics
total_sentences = sum(
    len(split_sentences(log["raw_summary"]))
    for log in all_halluc_logs
    if log["raw_summary"]
)
total_flagged = sum(
    len(log["hallucinations"])
    for log in all_halluc_logs
)
regen_accepted = sum(
    sum(1 for h in log["hallucinations"] if h["regen_accepted"])
    for log in all_halluc_logs
)
dropped      = total_flagged - regen_accepted
halluc_rate  = total_flagged / total_sentences if total_sentences > 0 else 0

# print results table
print("=" * 52)
print(f"{'Metric':<28} {'Baseline':>10} {'Verified':>10}")
print("=" * 52)
print(f"{'ROUGE-1 F1':<28} {raw_metrics['rouge1f']:>10} {final_metrics['rouge1f']:>10}")
print(f"{'ROUGE-L F1':<28} {raw_metrics['rougeLf']:>10} {final_metrics['rougeLf']:>10}")
print(f"{'ROUGE-1 Recall':<28} {raw_metrics['rouge1r']:>10} {final_metrics['rouge1r']:>10}")
print("=" * 52)
print(f"{'Total sentences':<28} {total_sentences:>21}")
print(f"{'Flagged sentences':<28} {total_flagged:>21}")
print(f"{'Hallucination rate':<28} {halluc_rate:>20.2%}")
print(f"{'Regenerated accepted':<28} {regen_accepted:>21}")
print(f"{'Dropped':<28} {dropped:>21}")
print("=" * 52)

# save results to Drive
results = {
    "raw_metrics":        raw_metrics,
    "verified_metrics":   final_metrics,
    "hallucination_rate": round(halluc_rate, 4),
    "total_sentences":    total_sentences,
    "total_flagged":      total_flagged,
    "regen_accepted":     regen_accepted,
    "dropped":            dropped,
}

os.makedirs(path_output, exist_ok=True)

with open(f"{path_output}metrics_verified.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics_verified.json saved")

with open(f"{path_output}halluc_log.json", "w") as f:
    json.dump(all_halluc_logs, f, indent=2)
print("halluc_log.json saved")

with open(f"{path_output}final_predictions.json", "w") as f:
    json.dump(final_summaries, f, indent=2)
print("final_predictions.json saved")

# verify files exist and show sizes
print("\nFiles saved:")
for fname in ["metrics_verified.json", "halluc_log.json", "final_predictions.json"]:
    fpath = f"{path_output}{fname}"
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  {fname}  ({size:,} bytes)")

Metric                         Baseline   Verified
ROUGE-1 F1                      38.8941    38.8691
ROUGE-L F1                      24.2784     24.444
ROUGE-1 Recall                  47.2585    45.9837
Total sentences                               1004
Flagged sentences                              132
Hallucination rate                         13.15%
Regenerated accepted                            80
Dropped                                         52

metrics_verified.json saved
halluc_log.json saved
final_predictions.json saved

Files saved:
  metrics_verified.json  (320 bytes)
  halluc_log.json  (598,459 bytes)
  final_predictions.json  (205,099 bytes)


In [ ]:
#@title Faithfulness evaluation using BERTScore

from bert_score import score as bert_score
import numpy as np

print("Computing BERTScore for raw summaries vs source articles...")
sources = [article["trunc_input"] for article in scored_dataset]

# BERTScore: raw summaries vs source articles
P_raw, R_raw, F_raw = bert_score(
    cands=raw_summaries,
    refs=sources,
    lang="en",
    model_type="distilbert-base-uncased",
    device=device,
    verbose=True
)

# BERTScore: verified summaries vs source articles
print("\nComputing BERTScore for verified summaries vs source articles...")
verified_clean = [s if s and s.strip() else "empty" for s in final_summaries]

P_ver, R_ver, F_ver = bert_score(
    cands=verified_clean,
    refs=sources,
    lang="en",
    model_type="distilbert-base-uncased",
    device=device,
    verbose=True
)

raw_f   = round(F_raw.mean().item()  * 100, 4)
ver_f   = round(F_ver.mean().item()  * 100, 4)
raw_p   = round(P_raw.mean().item()  * 100, 4)
ver_p   = round(P_ver.mean().item()  * 100, 4)
raw_r   = round(R_raw.mean().item()  * 100, 4)
ver_r   = round(R_ver.mean().item()  * 100, 4)

print("\n" + "=" * 56)
print(f"{'BERTScore Metric':<28} {'Baseline':>12} {'Verified':>12}")
print("=" * 56)
print(f"{'Precision':<28} {raw_p:>12} {ver_p:>12}")
print(f"{'Recall':<28} {raw_r:>12} {ver_r:>12}")
print(f"{'F1 (faithfulness)':<28} {raw_f:>12} {ver_f:>12}")
print("=" * 56)
print(f"{'Change':<28} {'':>12} {ver_f - raw_f:>+11.4f}")
print("=" * 56)

# save results
bertscore_results = {
    "raw": {
        "precision": raw_p,
        "recall":    raw_r,
        "f1":        raw_f,
    },
    "verified": {
        "precision": ver_p,
        "recall":    ver_r,
        "f1":        ver_f,
    },
    "f1_change": round(ver_f - raw_f, 4)
}

with open(f"{path_output}bertscore_faithfulness.json", "w") as f:
    json.dump(bertscore_results, f, indent=2)
print(f"\nBERTScore results saved to {path_output}bertscore_faithfulness.json")

Computing BERTScore for raw summaries vs source articles...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 12.50 seconds, 40.00 sentences/sec

BERTScore Metric                 Baseline     Verified
Precision                         85.3149      85.3368
Recall                            73.8647      73.8001
F1 (faithfulness)                 79.1469      79.1184
Change                                        -0.0285

BERTScore results saved to /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/hallucination_extension/cnn_verified_predictions/bertscore_faithfulness.json


In [ ]:
#@title Sentence-Level Faithfulness Analysis
from sentence_transformers import util
import numpy as np

print("Computing sentence-level faithfulness for flagged sentences...")

improvements    = []   # cases where verified sentence is more faithful
degradations    = []   # cases where verified sentence is less faithful
unchanged       = []   # cases where score did not change

for log in all_halluc_logs:
    for h in log["hallucinations"]:

        if not h["regen_accepted"] or not h["regenerated"]:
            continue

        original    = h["original"]
        regenerated = h["regenerated"]
        evidence    = h["evidence"]

        if not evidence:
            continue

        # encode original, regenerated and evidence
        orig_emb  = embedder.encode(original,    convert_to_tensor=True)
        regen_emb = embedder.encode(regenerated, convert_to_tensor=True)
        ev_emb    = embedder.encode(evidence,    convert_to_tensor=True)

        # compute similarity of original vs evidence
        orig_sim  = float(util.cos_sim(orig_emb,  ev_emb).max())

        # compute similarity of regenerated vs evidence
        regen_sim = float(util.cos_sim(regen_emb, ev_emb).max())

        diff = regen_sim - orig_sim

        if diff > 0.01:
            improvements.append({
                "original":    original,
                "regenerated": regenerated,
                "orig_sim":    round(orig_sim,  4),
                "regen_sim":   round(regen_sim, 4),
                "improvement": round(diff, 4)
            })
        elif diff < -0.01:
            degradations.append({
                "original":    original,
                "regenerated": regenerated,
                "orig_sim":    round(orig_sim,  4),
                "regen_sim":   round(regen_sim, 4),
                "degradation": round(diff, 4)
            })
        else:
            unchanged.append(diff)

total = len(improvements) + len(degradations) + len(unchanged)

print("\n" + "=" * 56)
print(f"{'Sentence-level Faithfulness Analysis'}")
print("=" * 56)
print(f"{'Total regenerated sentences:':<35} {total:>6}")
print(f"{'Improved (more faithful):':<35} {len(improvements):>6}  ({len(improvements)/total*100:.1f}%)")
print(f"{'Degraded (less faithful):':<35} {len(degradations):>6}  ({len(degradations)/total*100:.1f}%)")
print(f"{'Unchanged:':<35} {len(unchanged):>6}  ({len(unchanged)/total*100:.1f}%)")
print("=" * 56)

if improvements:
    avg_imp = np.mean([x["improvement"] for x in improvements])
    print(f"\nAverage similarity improvement: +{avg_imp:.4f}")

if degradations:
    avg_deg = np.mean([x["degradation"] for x in degradations])
    print(f"Average similarity degradation: {avg_deg:.4f}")

# show 3 best examples
print("\n=== Top 3 most improved sentences ===")
improvements.sort(key=lambda x: x["improvement"], reverse=True)
for i, ex in enumerate(improvements[:3]):
    print(f"\nExample {i+1}:")
    print(f"  Original:    {ex['original'][:80]}...")
    print(f"  Regenerated: {ex['regenerated'][:80]}...")
    print(f"  Similarity:  {ex['orig_sim']} → {ex['regen_sim']} (+{ex['improvement']})")

# save results
sentence_results = {
    "total_regenerated":  total,
    "improved":           len(improvements),
    "degraded":           len(degradations),
    "unchanged":          len(unchanged),
    "improved_pct":       round(len(improvements)/total*100, 2) if total > 0 else 0,
    "degraded_pct":       round(len(degradations)/total*100, 2) if total > 0 else 0,
    "avg_improvement":    round(np.mean([x["improvement"] for x in improvements]), 4) if improvements else 0,
    "top_examples":       improvements[:10],
}

with open(f"{path_output}sentence_faithfulness.json", "w") as f:
    json.dump(sentence_results, f, indent=2)
print(f"\nResults saved to {path_output}sentence_faithfulness.json")

Computing sentence-level faithfulness for flagged sentences...

Sentence-level Faithfulness Analysis
Total regenerated sentences:            11
Improved (more faithful):               11  (100.0%)
Degraded (less faithful):                0  (0.0%)
Unchanged:                               0  (0.0%)

Average similarity improvement: +0.2002

=== Top 3 most improved sentences ===

Example 1:
  Original:    The tightly controlled state media in North Korea suppresses such outside influe...
  Regenerated: The regime hates this film because it shows Kim Jong Un as a man, not a God, and...
  Similarity:  0.4602 → 0.8193 (+0.3591)

Example 2:
  Original:    The vandalism incident, which occurred in broad daylight, was captured in a phot...
  Regenerated: The LAPD has requested the public's help in identifying the person responsible a...
  Similarity:  0.5348 → 0.818 (+0.2832)

Example 3:
  Original:    The unnamed woman, who had mental health struggles, was identified by witnesses ...
  Regener